# ROBERT UI Notebook Launcher

Use this notebook to install UI dependencies, launch the local Dash app, and stop it cleanly.

This notebook is intended for users who are comfortable running Jupyter notebooks and want a guided way to run the app.

## Step 1: Configure Paths and Environment

This cell resolves the project root and confirms that required files exist.

Expected output:
- The project root path
- Confirmation that `start_ui.py` and `requirements_ui.txt` were found

In [1]:
from pathlib import Path
import os

# Resolve project root by walking up from this notebook location.
possible_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path.cwd().parent.parent.parent,
]

PROJECT_ROOT = None
for candidate in possible_roots:
    if (candidate / 'AGENTS.md').exists() and (candidate / 'start_ui.py').exists():
        PROJECT_ROOT = candidate.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate project root containing AGENTS.md and start_ui.py')

START_UI = PROJECT_ROOT / 'start_ui.py'
REQS_UI = PROJECT_ROOT / 'requirements_ui.txt'

print(f'Project root: {PROJECT_ROOT}')
print(f'start_ui.py found: {START_UI.exists()} -> {START_UI}')
print(f'requirements_ui.txt found: {REQS_UI.exists()} -> {REQS_UI}')
print(f'ROBERT_CHAT_API_KEY set: {bool(os.getenv("ROBERT_CHAT_API_KEY"))}')

Project root: /Users/cjcscha/ROBERT/helper_rob/robert
start_ui.py found: True -> /Users/cjcscha/ROBERT/helper_rob/robert/start_ui.py
requirements_ui.txt found: True -> /Users/cjcscha/ROBERT/helper_rob/robert/requirements_ui.txt
ROBERT_CHAT_API_KEY set: False


## Step 2: Install/Update UI Dependencies

Run this cell to install dependencies from `requirements_ui.txt` into the active notebook Python environment.

You can skip this if already installed, but running it is safe.

In [2]:
import subprocess
import sys

cmd = [sys.executable, '-m', 'pip', 'install', '-r', str(REQS_UI)]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=False)
print('Exit code:', result.returncode)
if result.returncode != 0:
    raise RuntimeError('Dependency installation failed. See output above.')

Running: /Users/cjcscha/mambaforge/envs/robert/bin/python -m pip install -r /Users/cjcscha/ROBERT/helper_rob/robert/requirements_ui.txt
Exit code: 0



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Step 3: (Optional) Set API Key for This Notebook Session

Set your OpenAI API key only when you are ready to test chat fallback behavior.

If left blank, the app still runs and displays diagnostics, but chat remains non-functional.

In [3]:
# Paste your key between quotes only when needed.
OPENAI_KEY = ''

if OPENAI_KEY.strip():
    os.environ['ROBERT_CHAT_API_KEY'] = OPENAI_KEY.strip()
    print('ROBERT_CHAT_API_KEY set for current notebook session.')
else:
    print('No API key set in this cell. UI will still launch.')

No API key set in this cell. UI will still launch.


## Step 4: Start the UI Server

This starts `start_ui.py` in the background and prints startup logs.

Expected output includes a URL like `http://127.0.0.1:8050`.

In [4]:
import subprocess
import sys
import time

if 'ui_proc' in globals() and ui_proc is not None and ui_proc.poll() is None:
    raise RuntimeError('UI server appears to already be running. Stop it first using the stop cell.')

ui_proc = subprocess.Popen(
    [sys.executable, str(START_UI)],
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

print(f'Started UI process with PID: {ui_proc.pid}')
print('Streaming startup logs for up to 15 seconds...')

deadline = time.time() + 15
while time.time() < deadline:
    line = ui_proc.stdout.readline()
    if line:
        print(line.rstrip())
        if 'Running on http://127.0.0.1:8050' in line or 'Dash is running on http://127.0.0.1:8050/' in line:
            print('UI appears ready.')
            break
    if ui_proc.poll() is not None:
        raise RuntimeError(f'UI process exited early with code {ui_proc.returncode}')
else:
    print('Startup window ended. If no errors were shown, try opening the URL below.')

UI_URL = 'http://127.0.0.1:8050'
print('Open in browser:', UI_URL)

Started UI process with PID: 76224
Streaming startup logs for up to 15 seconds...
[2026-05-16 14:24:46,067] WARNING: ROBERT_CHAT_API_KEY not found. Set via environment variable or create config/.env. Chat functionality will not work.
[2026-05-16 14:24:46,067] INFO: Project root: /Users/cjcscha/ROBERT/helper_rob/robert
[2026-05-16 14:24:46,067] INFO: Run archive: /Users/cjcscha/ROBERT/helper_rob/robert/agent/run_archive
[2026-05-16 14:24:46,067] WARNING: API key not configured. Chat functionality will not work. Set ROBERT_CHAT_API_KEY environment variable or create config/.env
[2026-05-16 14:24:46,076] INFO: Found 2 diagnostic runs
[2026-05-16 14:24:46,077] INFO: Starting app on 127.0.0.1:8050
[2026-05-16 14:24:46,077] INFO: Open http://127.0.0.1:8050 in your browser
[2026-05-16 14:24:46,077] INFO: Press Ctrl+C to stop
Dash is running on http://127.0.0.1:8050/
UI appears ready.
Open in browser: http://127.0.0.1:8050


## Step 5: Open the UI in Your Browser

This attempts to open the local app automatically.

In [5]:
import webbrowser

url = globals().get('UI_URL', 'http://127.0.0.1:8050')
opened = webbrowser.open(url)
print('Attempted to open:', url)
print('Browser open call returned:', opened)

Attempted to open: http://127.0.0.1:8050
Browser open call returned: True


## Step 6: Stop the UI Server

Run this cell when you are done using the app.

It safely terminates the background process started in Step 4.

In [6]:
if 'ui_proc' in globals() and ui_proc is not None and ui_proc.poll() is None:
    ui_proc.terminate()
    try:
        ui_proc.wait(timeout=5)
        print(f'UI process {ui_proc.pid} terminated with code {ui_proc.returncode}')
    except Exception:
        ui_proc.kill()
        print(f'UI process {ui_proc.pid} force-killed')
else:
    print('No running UI process found in this notebook session.')

No running UI process found in this notebook session.


## Troubleshooting

- **Port in use (8050)**: Stop any previous server process, then rerun Step 4.
- **No runs found**: Ensure archived runs with `run_context.json` and `diagnosis_summary.md` exist in `agent/run_archive/`.
- **Chat unavailable**: This is expected without `ROBERT_CHAT_API_KEY` set.